# RetailMind AI — Day 13: Classical Review Sentiment

This standalone notebook cleans Amazon review text with NLTK, trains a TF-IDF Logistic Regression baseline, evaluates sentiment, and shows informative terms.

## Before starting

Use Amazon Polarity `train.csv` (headerless `label,title,content` rows are supported) or a Kaggle Amazon reviews CSV (`Score`, `Summary`, `Text`). Put it in `data/amazon-reviews/` or set `DATA_CSV` below. The notebook keeps a balanced sample of at most 3,000 reviews per sentiment.

In [ ]:
from pathlib import Path
import csv
import sys
import joblib
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_CSV = None  # Or set this to a review CSV path.
DATA_OPTIONS = [ROOT / "data" / "amazon-reviews" / "train.csv", ROOT / "data" / "amazon_polarity" / "train.csv", ROOT / "data" / "Reviews.csv"]
if DATA_CSV: csv_path = Path(DATA_CSV)
else: csv_path = next((path for path in DATA_OPTIONS if path.is_file()), None)
if csv_path is None or not csv_path.is_file(): raise FileNotFoundError("Put an Amazon review CSV under data/amazon-reviews/ or set DATA_CSV.")
MAX_PER_CLASS, SEED = 3000, 42
try: STOP_WORDS = set(stopwords.words("english")) - {"no", "not", "nor"}
except LookupError as error: raise RuntimeError("Run python -c \"import nltk; nltk.download('stopwords')\" first.") from error
TOKENIZER = RegexpTokenizer(r"[a-z]+")
def clean_text(text):
    return " ".join(token for token in TOKENIZER.tokenize(str(text).lower()) if token not in STOP_WORDS)
print("Dataset:", csv_path)

## Load, label, and clean reviews

Amazon Polarity uses labels 0/1 or 1/2 depending on the CSV export. Star ratings 1–2 are negative, 4–5 positive, and neutral 3-star reviews are skipped.

In [ ]:
from src.day13_review_sentiment import load_reviews
raw_texts, labels = load_reviews(csv_path, MAX_PER_CLASS)
print("Balanced sample:", {"negative": labels.count(0), "positive": labels.count(1)})
texts = [clean_text(text) for text in raw_texts]
x_train, x_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=SEED, stratify=labels)

## TF-IDF and Logistic Regression

TF-IDF gives more weight to useful words and phrases. Logistic Regression learns which terms are associated with each sentiment.

In [ ]:
vectorizer = TfidfVectorizer(lowercase=False, ngram_range=(1, 2), max_features=20000, min_df=2, sublinear_tf=True)
train_features = vectorizer.fit_transform(x_train)
test_features = vectorizer.transform(x_test)
classifier = LogisticRegression(max_iter=500, random_state=SEED)
classifier.fit(train_features, y_train)
predictions = classifier.predict(test_features)
accuracy = accuracy_score(y_test, predictions)
report = classification_report(y_test, predictions, labels=[0, 1], target_names=["negative", "positive"], output_dict=True, zero_division=0)
print(f"Test accuracy: {accuracy:.3f}")
print(classification_report(y_test, predictions, labels=[0, 1], target_names=["negative", "positive"], zero_division=0))

## Find useful terms and save the module

Positive model weights indicate positive sentiment; negative weights indicate negative sentiment. The saved vectorizer and classifier can be loaded to score future reviews.

In [ ]:
MODEL_DIR, REPORT_DIR = ROOT / "models", ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "day13_review_sentiment_notebook.joblib"
joblib.dump({"vectorizer": vectorizer, "classifier": classifier, "labels": ("negative", "positive"), "stop_words": list(STOP_WORDS)}, model_path)
with (REPORT_DIR / "day13_notebook_metrics.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["dataset", "train_reviews", "test_reviews", "accuracy", "negative_precision", "negative_recall", "positive_precision", "positive_recall"])
    writer.writeheader()
    writer.writerow({"dataset": csv_path.name, "train_reviews": len(y_train), "test_reviews": len(y_test), "accuracy": accuracy, "negative_precision": report["negative"]["precision"], "negative_recall": report["negative"]["recall"], "positive_precision": report["positive"]["precision"], "positive_recall": report["positive"]["recall"]})
terms, coefficients = vectorizer.get_feature_names_out(), classifier.coef_[0]
negative = sorted(range(len(terms)), key=coefficients.__getitem__)[:20]
positive = sorted(range(len(terms)), key=coefficients.__getitem__, reverse=True)[:20]
with (REPORT_DIR / "day13_notebook_informative_terms.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["sentiment", "term", "coefficient"])
    writer.writeheader()
    for sentiment, indices in (("positive", positive), ("negative", negative)):
        for index in indices: writer.writerow({"sentiment": sentiment, "term": terms[index], "coefficient": float(coefficients[index])})
print("Saved model:", model_path)
print("Saved test metrics and informative terms under:", REPORT_DIR)

# Try a new review after fitting.
example = clean_text("Great quality and fast delivery.")
probabilities = classifier.predict_proba(vectorizer.transform([example]))[0]
best = int(probabilities.argmax())
print("Example prediction:", ("negative", "positive")[best], "confidence:", round(float(probabilities[best]), 3))

## Saved files

- Model: `models/day13_review_sentiment_notebook.joblib`
- Evaluation: `reports/day13_notebook_metrics.csv`
- Informative terms: `reports/day13_notebook_informative_terms.csv`